In [0]:
%run /Workspace/Users/parbhavt@publicisgroupe.net/Project/00_Config_Setup

In [0]:
STORAGE_PATH = "abfss://raw@rxmaxdetestsa.dfs.core.windows.net/personal-learning/parth/retailpulse/raw"

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Bronze Ingestion
# MAGIC Reads raw JSON from landing zone, writes to Bronze Delta tables.
# MAGIC - Schema-on-read (no enforcement)
# MAGIC - Adds metadata columns
# MAGIC - Idempotent (re-running same date doesn't duplicate)
# MAGIC - Partitioned by ingestion_date

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, lit, input_file_name, expr
import uuid

# COMMAND ----------

# CONFIG
RAW_PATH    = "abfss://raw@rxmaxdetestsa.dfs.core.windows.net/personal-learning/parth/retailpulse/raw"
BRONZE_PATH = "abfss://raw@rxmaxdetestsa.dfs.core.windows.net/personal-learning/parth/retailpulse/bronze"

# Process specific date — pass via widget for parameterization
dbutils.widgets.text("ingestion_date", "2026-04-25", "Ingestion Date (YYYY-MM-DD)")
INGESTION_DATE = dbutils.widgets.get("ingestion_date")

print(f"Processing date: {INGESTION_DATE}")

# COMMAND ----------

def add_bronze_metadata(df, ingestion_date):
    """Add audit columns to every Bronze table"""
    ingestion_id = str(uuid.uuid4())
    return (df
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", input_file_name())
        .withColumn("_ingestion_id", lit(ingestion_id))
        .withColumn("ingestion_date", lit(ingestion_date).cast("date"))
    )

def write_bronze(df, table_name, mode="append"):
    """Write to Bronze Delta with partitioning"""
    path = f"{BRONZE_PATH}/{table_name}"
    (df.write
        .format("delta")
        .mode(mode)
        .partitionBy("ingestion_date")
        .option("mergeSchema", "true")   # handles schema drift
        .save(path))
    print(f"  → Wrote {df.count()} rows to {path}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Idempotency: delete partition if rerunning same date

# COMMAND ----------

from delta.tables import DeltaTable

def delete_partition_if_exists(table_path, ingestion_date):
    """Remove existing partition for this date so re-runs don't duplicate"""
    if DeltaTable.isDeltaTable(spark, table_path):
        dt = DeltaTable.forPath(spark, table_path)
        dt.delete(f"ingestion_date = '{ingestion_date}'")
        print(f"  Cleared existing partition for {ingestion_date} in {table_path}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Ingest Orders (date-partitioned source)

# COMMAND ----------

orders_source = f"{RAW_PATH}/orders/{INGESTION_DATE}/"
print(f"Reading: {orders_source}")

orders_raw = (spark.read
    .option("multiLine", "false")     # newline-delimited JSON
    .json(orders_source))

orders_bronze = add_bronze_metadata(orders_raw, INGESTION_DATE)

delete_partition_if_exists(f"{BRONZE_PATH}/orders", INGESTION_DATE)
write_bronze(orders_bronze, "orders")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Ingest Customers (full snapshot — overwrite each run)

# COMMAND ----------

customers_source = f"{RAW_PATH}/customers/"
print(f"Reading: {customers_source}")

customers_raw = spark.read.json(customers_source)
customers_bronze = add_bronze_metadata(customers_raw, INGESTION_DATE)

delete_partition_if_exists(f"{BRONZE_PATH}/customers", INGESTION_DATE)
write_bronze(customers_bronze, "customers")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Ingest Products (full snapshot)

# COMMAND ----------

products_source = f"{RAW_PATH}/products/"
print(f"Reading: {products_source}")

products_raw = spark.read.json(products_source)
products_bronze = add_bronze_metadata(products_raw, INGESTION_DATE)

delete_partition_if_exists(f"{BRONZE_PATH}/products", INGESTION_DATE)
write_bronze(products_bronze, "products")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Verify

# COMMAND ----------

print("=== ORDERS ===")
spark.read.format("delta").load(f"{BRONZE_PATH}/orders").show(3, truncate=False)
print(f"Total: {spark.read.format('delta').load(f'{BRONZE_PATH}/orders').count()}")

print("\n=== CUSTOMERS ===")
spark.read.format("delta").load(f"{BRONZE_PATH}/customers").show(3, truncate=False)

print("\n=== PRODUCTS ===")
spark.read.format("delta").load(f"{BRONZE_PATH}/products").show(3, truncate=False)

Processing date: 2026-04-25
Reading: abfss://raw@rxmaxdetestsa.dfs.core.windows.net/personal-learning/parth/retailpulse/raw/orders/2026-04-25/
  → Wrote 10100 rows to abfss://raw@rxmaxdetestsa.dfs.core.windows.net/personal-learning/parth/retailpulse/bronze/orders
Reading: abfss://raw@rxmaxdetestsa.dfs.core.windows.net/personal-learning/parth/retailpulse/raw/customers/
  → Wrote 1000 rows to abfss://raw@rxmaxdetestsa.dfs.core.windows.net/personal-learning/parth/retailpulse/bronze/customers
Reading: abfss://raw@rxmaxdetestsa.dfs.core.windows.net/personal-learning/parth/retailpulse/raw/products/
  → Wrote 500 rows to abfss://raw@rxmaxdetestsa.dfs.core.windows.net/personal-learning/parth/retailpulse/bronze/products
=== ORDERS ===
+-----------+------------------------------------+-------------------+--------------+----------+--------+---------+--------------------------+--------------------------------------------------------------------------------------------------------------------+-----

In [0]:
# Replace the widget cell with this for now
DATES_TO_PROCESS = ["2026-04-25", "2026-04-26", "2026-04-27", "2026-04-28",
                    "2026-04-29", "2026-04-30", "2026-05-01"]

In [0]:
def ingest_date(ingestion_date):
    print(f"\n{'='*50}\nProcessing {ingestion_date}\n{'='*50}")
    
    # Orders
    orders_source = f"{RAW_PATH}/orders/{ingestion_date}/"
    orders_raw = spark.read.json(orders_source)
    orders_bronze = add_bronze_metadata(orders_raw, ingestion_date)
    delete_partition_if_exists(f"{BRONZE_PATH}/orders", ingestion_date)
    write_bronze(orders_bronze, "orders")
    
    # Customers
    customers_raw = spark.read.json(f"{RAW_PATH}/customers/")
    customers_bronze = add_bronze_metadata(customers_raw, ingestion_date)
    delete_partition_if_exists(f"{BRONZE_PATH}/customers", ingestion_date)
    write_bronze(customers_bronze, "customers")
    
    # Products
    products_raw = spark.read.json(f"{RAW_PATH}/products/")
    products_bronze = add_bronze_metadata(products_raw, ingestion_date)
    delete_partition_if_exists(f"{BRONZE_PATH}/products", ingestion_date)
    write_bronze(products_bronze, "products")

# Run for all dates
for d in DATES_TO_PROCESS:
    ingest_date(d)

print("\n✅ All dates processed")


Processing 2026-04-25
  Cleared existing partition for 2026-04-25 in abfss://raw@rxmaxdetestsa.dfs.core.windows.net/personal-learning/parth/retailpulse/bronze/orders
  → Wrote 10100 rows to abfss://raw@rxmaxdetestsa.dfs.core.windows.net/personal-learning/parth/retailpulse/bronze/orders
  Cleared existing partition for 2026-04-25 in abfss://raw@rxmaxdetestsa.dfs.core.windows.net/personal-learning/parth/retailpulse/bronze/customers
  → Wrote 1000 rows to abfss://raw@rxmaxdetestsa.dfs.core.windows.net/personal-learning/parth/retailpulse/bronze/customers
  Cleared existing partition for 2026-04-25 in abfss://raw@rxmaxdetestsa.dfs.core.windows.net/personal-learning/parth/retailpulse/bronze/products
  → Wrote 500 rows to abfss://raw@rxmaxdetestsa.dfs.core.windows.net/personal-learning/parth/retailpulse/bronze/products

Processing 2026-04-26
  Cleared existing partition for 2026-04-26 in abfss://raw@rxmaxdetestsa.dfs.core.windows.net/personal-learning/parth/retailpulse/bronze/orders
  → Wrot

In [0]:
df = spark.read.format("delta").load(f"{BRONZE_PATH}/orders")
df.groupBy("ingestion_date").count().orderBy("ingestion_date").show()
df.printSchema()

+--------------+-----+
|ingestion_date|count|
+--------------+-----+
|    2026-04-25|10100|
|    2026-04-26|10100|
|    2026-04-27|10100|
|    2026-04-28|10100|
|    2026-04-29|10100|
|    2026-04-30|10100|
|    2026-05-01|10100|
+--------------+-----+

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_timestamp: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- status: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_id: string (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- discount_code: string (nullable = true)

